# 9장 (실습 2): Trace ID로 요청 하나를 끝까지 추적해보기

> 책 『AI 보안, 클라우드에서 시작한다』 9장 [실습] 2번을 노트북에서 단계별로 실행하는 버전입니다. 모델 호출, 문서 검색, Tool 실행을 흉내 내는 작은 API 서버에서 요청마다 Trace ID를 붙여 요청 하나의 전체 흐름을 재구성하고, 로그에 남기기 전에 민감정보를 가리는 마스킹까지 확인합니다. 소요 시간은 30~40분입니다.
> 같은 폴더의 [README.md](./README.md)와 내용이 같고, 셀을 위에서부터 차례로 실행하면 됩니다.

## 🎯 학습 목표
1. 동시에 들어온 요청의 로그가 **뒤섞이는** 것을 보고, Trace ID 하나로 요청 하나의 흐름(검색 문서, Tool, 모델 호출)을 재구성합니다.
2. 민감정보가 게이트웨이 로그뿐 아니라 **검색어 로그에도** 남는다는 것을 보고, 모든 로그가 지나가는 **한 곳에서** 가리는 이유를 이해합니다.
3. 서비스 사이로 Trace ID를 **이어 주는** 방법을 확인합니다.

## 📝 실습 구성

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 설치 | 예정 |
| 1 | 서버 코드 살펴보고 실행하기 | 예정 |
| 2 | 여러 요청을 동시에 보내기 | 예정 |
| 3 | 로그 보기: Trace ID가 없다면? | 예정 |
| 4 | Trace ID로 요청 하나 재구성하기 | 예정 |
| 5 | 로그에 남기기 전에 민감정보 가리기 | 예정 |
| 6 | 서비스 사이로 Trace ID 이어 주기 | 예정 |
| 7 | 체크포인트와 과제 | 예정 |
| 8 | 정리 | 예정 |

## 실습 시나리오

AI 서비스 요청 하나는 Gateway → RAG 검색 → Agent 계획 → Tool 실행 → 모델 호출처럼 여러 단계를 지납니다. 요청이 동시에 여러 개 들어오면 로그가 뒤섞여 "이 줄이 어느 요청의 것인지" 알 수 없게 됩니다. 사고가 났을 때 "어떤 사용자의 어떤 질문이, 어떤 문서와 Tool을 거쳐, 무엇을 했는가"에 답하려면 무엇이 필요할까요?

![그림 실습9-2](../docs/images/그림실습9-2_TraceID_마스킹.png)


> ✅ **실습 전 확인사항**
>
> **1. 실행 환경**<br>
> - 이 노트북은 **리눅스(WSL2의 Ubuntu 포함)에서 Docker를 직접 부를 수 있는 환경**을 가정합니다. 셀 안의 `%%bash`는 그 셀 전체를 bash 명령으로 실행하라는 뜻입니다.
> - Python 3와 인터넷 연결(패키지 설치)이 필요합니다. 실제 AI 모델은 쓰지 않으므로 API Key와 Docker가 필요 없습니다. 8000 포트가 비어 있어야 합니다.
>
> **2. 노트북 열기**<br>
> 처음이라면 [SETUP.md](../SETUP.md)의 6단계(VS Code와 Jupyter 노트북 연결)를 먼저 한 번 해 둡니다. 그다음에는 아래 두 방법 가운데 하나로 이 노트북을 엽니다.
> - **VS Code**: Ubuntu 터미널에서 `cd ~/ai-cloud-security-labs && code .`를 실행합니다. 창 왼쪽 아래에 `WSL: Ubuntu-24.04`가 보이면, 왼쪽 탐색기에서 `ch09-ops-tracing-sw-inventory` 폴더를 펼쳐 이 노트북을 엽니다. 그리고 오른쪽 위 커널 표시를 눌러 **Select Another Kernel...** → **Python Environments...** → `.venv-lab`을 고릅니다. 목록에 `.venv-lab`이 없으면 6단계 ⑥의 2번처럼 `Python: Select Interpreter` → **Enter interpreter path...**로 `~/.venv-lab/bin/python`의 위치를 먼저 알려 줍니다.
> - **웹 브라우저(Jupyter Lab)**: Ubuntu 터미널에서 `cd ~/ai-cloud-security-labs/ch09-ops-tracing-sw-inventory && ~/.venv-lab/bin/jupyter lab`을 실행하고, 화면에 나온 `http://localhost:8888/...` 주소를 Windows의 브라우저에서 엽니다.
> - 노트북 파일만 따로 내려받거나 다른 폴더로 옮기지 마세요. 노트북은 자기가 들어 있는 `ch09-ops-tracing-sw-inventory` 폴더의 실습 파일을 읽습니다. 위 두 방법처럼 저장소 안에서 열면 따로 신경 쓸 것은 없습니다.
>
> **3. 기준 버전**<br>
> - 아래 출력은 2026년 10월 23일 Ubuntu 24.04(WSL2), Python 3.12와 그 시점의 FastAPI·uvicorn 최신 버전에서 실행한 결과입니다. 시각, 번호, ID 같은 값은 실행할 때마다 다릅니다. 직접 실행할 때는 출력이 없는 `ch09_lab2_trace_id_masking.ipynb`을 쓰세요.

## 0. 실습 준비와 설치

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 설치 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 1 | 서버 코드 살펴보고 실행하기 | 예정 |
| 2 | 여러 요청을 동시에 보내기 | 예정 |
| 3 | 로그 보기: Trace ID가 없다면? | 예정 |
| 4 | Trace ID로 요청 하나 재구성하기 | 예정 |
| 5 | 로그에 남기기 전에 민감정보 가리기 | 예정 |
| 6 | 서비스 사이로 Trace ID 이어 주기 | 예정 |
| 7 | 체크포인트와 과제 | 예정 |
| 8 | 정리 | 예정 |

In [1]:
import json
import os
import shutil
import subprocess
from pathlib import Path

from IPython.display import Markdown, display

LAB = Path.cwd()          # 이 노트북이 있는 폴더
os.environ["PATH"] = f"{Path.home() / 'bin'}:{os.environ['PATH']}"   # 실습 도구를 ~/bin에 둔 경우


def sh(cmd):
    """셸 명령을 실행하고 표준 출력을 문자열로 돌려줍니다."""
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()


def table(header, rows):
    """머리글과 행 목록으로 마크다운 표를 그립니다."""
    lines = ["| " + " | ".join(header) + " |", "|" + "---|" * len(header)]
    lines += ["| " + " | ".join(str(c) for c in r) + " |" for r in rows]
    display(Markdown("\n".join(lines)))


VERSION_CMD = {
    "python3": "python3 --version",
}
for tool, cmd in VERSION_CMD.items():
    state = f"✅ {sh(cmd).splitlines()[0] if sh(cmd) else ''}" if shutil.which(tool) else "❌ 없음"
    print(f"{tool:10} {state}")

python3    ✅ Python 3.12.3


실습 파일을 `~/trace-lab`으로 복사하고, 서버를 돌릴 가상환경 `~/trace-venv`에 FastAPI, uvicorn, httpx를 설치합니다. Ubuntu 24.04에는 `pip` 명령이 없고 시스템 Python에 패키지를 바로 설치하는 것도 막혀 있어서 가상환경을 씁니다.

> 📌 (참고) `python3 -m venv`가 `ensurepip` 오류를 내면 WSL 터미널에서 `sudo apt install -y python3-venv`를 먼저 실행합니다.

In [2]:
%%bash
rm -rf ~/trace-lab && cp -r trace ~/trace-lab
[ -x ~/trace-venv/bin/python ] || python3 -m venv ~/trace-venv
~/trace-venv/bin/pip install -q fastapi uvicorn httpx 2>&1 | grep -iv notice
~/trace-venv/bin/python -c 'import fastapi, uvicorn; print("FastAPI", fastapi.__version__, "/ uvicorn", uvicorn.__version__)'

FastAPI 0.161.0 / uvicorn 0.54.0


## 1. 서버 코드 살펴보고 실행하기

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 설치 | ✅ 완료 |
| 1 | 서버 코드 살펴보고 실행하기 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 2 | 여러 요청을 동시에 보내기 | 예정 |
| 3 | 로그 보기: Trace ID가 없다면? | 예정 |
| 4 | Trace ID로 요청 하나 재구성하기 | 예정 |
| 5 | 로그에 남기기 전에 민감정보 가리기 | 예정 |
| 6 | 서비스 사이로 Trace ID 이어 주기 | 예정 |
| 7 | 체크포인트와 과제 | 예정 |
| 8 | 정리 | 예정 |

`trace_app.py`의 핵심은 네 가지입니다.

| 코드 | 하는 일 | 9-2의 원칙 |
|---|---|---|
| `add_trace_id` 미들웨어 | 요청이 들어오면 Trace ID를 만들고, 앞단에서 받은 `X-Trace-Id`가 있으면 이어 쓴다. 응답 헤더로 돌려준다 | Trace ID 기반 End-to-End 추적 |
| `trace_id_var`와 `JsonFormatter` | 요청을 처리하는 동안 남기는 모든 로그에 같은 Trace ID를 자동으로 붙인다 | 통합 로깅 |
| `gateway`·`rag`·`agent`·`tool`·`model` 로거 | 요청 수신, 문서 검색, Agent 계획, Tool 호출, 모델 호출을 각각 기록한다 | Prompt·Data·Agent·Tool 로깅 |
| `mask()` | 로그 파일에 쓰기 직전에 주민등록번호, 카드번호, 휴대전화 번호, 이메일을 가린다 | 저장 전 마스킹 |

> 🔎 (용어 풀이) **미들웨어(middleware)**: 서버에 들어오는 모든 요청이 실제 처리 코드에 닿기 전에, 그리고 응답이 나가기 전에 거쳐 가는 공통 코드입니다. 건물 입구의 안내 데스크처럼 모든 방문자에게 출입증(Trace ID)을 달아 주는 일을 한 곳에서 합니다. 그래서 처리 코드마다 Trace ID를 만드는 코드를 넣지 않아도 됩니다.

In [3]:
WORK = Path.home() / "trace-lab"
src = (WORK / "trace_app.py").read_text()
print(src[src.index('@app.middleware("http")'):src.index("# 검색 대상 문서")])

@app.middleware("http")
async def add_trace_id(request: Request, call_next):
    # 앞단(다른 서비스)에서 받은 Trace ID가 있으면 이어 쓰고, 없으면 새로 만든다
    tid = request.headers.get("X-Trace-Id") or uuid.uuid4().hex[:12]
    trace_id_var.set(tid)
    response = await call_next(request)
    response.headers["X-Trace-Id"] = tid     # 응답에도 돌려주어, 사용자가 문의할 때 이 번호로 찾을 수 있게 한다
    return response




서버를 노트북 뒤에서 실행합니다. 터미널에서 `uvicorn trace_app:app --port 8000`을 실행하는 것과 같습니다. 5단계에서 마스킹을 끄고 다시 띄우므로 함수로 만들어 둡니다.

In [4]:
import socket
import time
import urllib.request

LOG = WORK / "app-log.jsonl"
server = None


def start_server(mask=True):
    """trace_app 서버를 (다시) 띄웁니다. mask=False면 LOG_MASK=off로 띄웁니다."""
    global server
    if server:
        server.terminate(); server.wait(timeout=5)
    env = dict(os.environ, LOG_MASK="on" if mask else "off")
    server = subprocess.Popen([str(Path.home() / "trace-venv/bin/uvicorn"), "trace_app:app", "--port", "8000"],
                              cwd=WORK, env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(50):
        try:
            socket.create_connection(("127.0.0.1", 8000), timeout=0.2).close()
            break
        except OSError:
            time.sleep(0.2)
    print(f"서버 실행 중: PID {server.pid}, 마스킹 {'켬' if mask else '끔'}")


def ask(user, question, order_id=None, trace=None):
    """/ask에 질문을 보내고 응답 헤더의 X-Trace-Id를 돌려줍니다."""
    body = {"user": user, "question": question, **({"order_id": order_id} if order_id else {})}
    headers = {"Content-Type": "application/json", **({"X-Trace-Id": trace} if trace else {})}
    req = urllib.request.Request("http://localhost:8000/ask", json.dumps(body).encode(), headers)
    with urllib.request.urlopen(req) as r:
        return r.headers["X-Trace-Id"]


def logs():
    return [json.loads(l) for l in LOG.read_text().splitlines()] if LOG.exists() else []


LOG.unlink(missing_ok=True)
start_server()

서버 실행 중: PID 117231, 마스킹 켬


## 2. 여러 요청을 동시에 보내기

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 설치 | ✅ 완료 |
| 1 | 서버 코드 살펴보고 실행하기 | ✅ 완료 |
| 2 | 여러 요청을 동시에 보내기 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 3 | 로그 보기: Trace ID가 없다면? | 예정 |
| 4 | Trace ID로 요청 하나 재구성하기 | 예정 |
| 5 | 로그에 남기기 전에 민감정보 가리기 | 예정 |
| 6 | 서비스 사이로 Trace ID 이어 주기 | 예정 |
| 7 | 체크포인트와 과제 | 예정 |
| 8 | 정리 | 예정 |

요청 세 개를 **동시에** 보냅니다. 터미널에서 `curl ... &`을 세 번 실행하는 것과 같습니다. 서버는 검색·Tool·모델 호출마다 일부러 무작위로 조금씩 기다리므로 세 요청의 처리가 서로 겹칩니다.

In [5]:
from concurrent.futures import ThreadPoolExecutor

REQUESTS = [("kim", "환불 되나요?", "A-1001"), ("lee", "배송 언제 와요?", "B-2002"), ("park", "교환하고 싶어요", "C-3003")]
with ThreadPoolExecutor(3) as pool:
    traces = list(pool.map(lambda r: ask(*r), REQUESTS))
TRACE = dict(zip([r[0] for r in REQUESTS], traces))
table(["사용자", "질문", "응답 헤더 X-Trace-Id"], [(r[0], r[1], TRACE[r[0]]) for r in REQUESTS])

| 사용자 | 질문 | 응답 헤더 X-Trace-Id |
|---|---|---|
| kim | 환불 되나요? | c46f87b1bb1b |
| lee | 배송 언제 와요? | a756caa67ea0 |
| park | 교환하고 싶어요 | 256e3593ddf0 |

요청마다 다른 Trace ID가 응답 헤더로 돌아옵니다.

## 3. 로그 보기: Trace ID가 없다면?

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 설치 | ✅ 완료 |
| 1 | 서버 코드 살펴보고 실행하기 | ✅ 완료 |
| 2 | 여러 요청을 동시에 보내기 | ✅ 완료 |
| 3 | 로그 보기: Trace ID가 없다면? | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 4 | Trace ID로 요청 하나 재구성하기 | 예정 |
| 5 | 로그에 남기기 전에 민감정보 가리기 | 예정 |
| 6 | 서비스 사이로 Trace ID 이어 주기 | 예정 |
| 7 | 체크포인트와 과제 | 예정 |
| 8 | 정리 | 예정 |

로그 파일을 그대로 봅니다. 단, **`trace_id` 열을 가리고** 봅니다.

In [6]:
table(["#", "시각", "단계", "메시지"], [(i, e["time"], e["step"], e["msg"]) for i, e in enumerate(logs(), 1)])

| # | 시각 | 단계 | 메시지 |
|---|---|---|---|
| 1 | 05:27:58 | gateway | 요청 수신 user=park question='교환하고 싶어요' |
| 2 | 05:27:58 | gateway | 요청 수신 user=lee question='배송 언제 와요?' |
| 3 | 05:27:58 | gateway | 요청 수신 user=kim question='환불 되나요?' |
| 4 | 05:27:58 | rag | 검색어='교환하고 싶어요' 결과=['교환정책.pdf#p2'] |
| 5 | 05:27:58 | agent | 계획: 문서 검색 → 주문 조회 Tool → 답변 생성 |
| 6 | 05:27:58 | rag | 검색어='배송 언제 와요?' 결과=['배송안내.pdf#p1'] |
| 7 | 05:27:58 | agent | 계획: 문서 검색 → 주문 조회 Tool → 답변 생성 |
| 8 | 05:27:58 | rag | 검색어='환불 되나요?' 결과=['환불규정.pdf#p3'] |
| 9 | 05:27:58 | agent | 계획: 문서 검색 → 주문 조회 Tool → 답변 생성 |
| 10 | 05:27:59 | tool | Tool 실행 name=get_order_status arg=B-2002 |
| 11 | 05:27:59 | tool | Tool 실행 name=get_order_status arg=C-3003 |
| 12 | 05:27:59 | tool | Tool 실행 name=get_order_status arg=A-1001 |
| 13 | 05:27:59 | model | 모델 호출 prompt_chars=63 tokens=152 |
| 14 | 05:27:59 | gateway | 응답 반환 |
| 15 | 05:27:59 | model | 모델 호출 prompt_chars=62 tokens=152 |
| 16 | 05:27:59 | gateway | 응답 반환 |
| 17 | 05:27:59 | model | 모델 호출 prompt_chars=61 tokens=152 |
| 18 | 05:27:59 | gateway | 응답 반환 |

세 요청의 로그 18줄이 **뒤섞여** 기록되어 있습니다. 같은 초에 들어온 요청이라 시간으로는 구분할 수 없고, 사용자 이름은 첫 줄(`요청 수신`)에만 있습니다.

> 🙋 (생각해 보기)
> 위 표만 보고 "lee의 질문이 어떤 Tool을 어떤 주문번호로 불렀는지" 답할 수 있나요? Tool 로그의 주문번호(`B-2002`)로 짐작할 수는 있지만, 모델 호출 줄은 어느 요청의 것인지 알 방법이 없습니다.

## 4. Trace ID로 요청 하나 재구성하기

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 설치 | ✅ 완료 |
| 1 | 서버 코드 살펴보고 실행하기 | ✅ 완료 |
| 2 | 여러 요청을 동시에 보내기 | ✅ 완료 |
| 3 | 로그 보기: Trace ID가 없다면? | ✅ 완료 |
| 4 | Trace ID로 요청 하나 재구성하기 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 5 | 로그에 남기기 전에 민감정보 가리기 | 예정 |
| 6 | 서비스 사이로 Trace ID 이어 주기 | 예정 |
| 7 | 체크포인트와 과제 | 예정 |
| 8 | 정리 | 예정 |

lee의 요청에 붙은 Trace ID 하나로 검색합니다. 터미널에서 `grep <Trace ID> app-log.jsonl`과 같습니다.

In [7]:
print(sh(f"grep {TRACE['lee']} {LOG}"))

{"time": "05:27:58", "trace_id": "a756caa67ea0", "step": "gateway", "msg": "요청 수신 user=lee question='배송 언제 와요?'"}
{"time": "05:27:58", "trace_id": "a756caa67ea0", "step": "rag", "msg": "검색어='배송 언제 와요?' 결과=['배송안내.pdf#p1']"}
{"time": "05:27:58", "trace_id": "a756caa67ea0", "step": "agent", "msg": "계획: 문서 검색 → 주문 조회 Tool → 답변 생성"}
{"time": "05:27:59", "trace_id": "a756caa67ea0", "step": "tool", "msg": "Tool 실행 name=get_order_status arg=B-2002"}
{"time": "05:27:59", "trace_id": "a756caa67ea0", "step": "model", "msg": "모델 호출 prompt_chars=63 tokens=152"}
{"time": "05:27:59", "trace_id": "a756caa67ea0", "step": "gateway", "msg": "응답 반환"}


In [8]:
table(["시각", "Trace ID", "단계", "메시지"],
      [(e["time"], e["trace_id"], e["step"], e["msg"]) for e in logs() if e["trace_id"] == TRACE["lee"]])

| 시각 | Trace ID | 단계 | 메시지 |
|---|---|---|---|
| 05:27:58 | a756caa67ea0 | gateway | 요청 수신 user=lee question='배송 언제 와요?' |
| 05:27:58 | a756caa67ea0 | rag | 검색어='배송 언제 와요?' 결과=['배송안내.pdf#p1'] |
| 05:27:58 | a756caa67ea0 | agent | 계획: 문서 검색 → 주문 조회 Tool → 답변 생성 |
| 05:27:59 | a756caa67ea0 | tool | Tool 실행 name=get_order_status arg=B-2002 |
| 05:27:59 | a756caa67ea0 | model | 모델 호출 prompt_chars=63 tokens=152 |
| 05:27:59 | a756caa67ea0 | gateway | 응답 반환 |

뒤섞인 18줄 가운데 6줄만 남고, 한 요청의 흐름이 "요청 수신 → 문서 검색(배송안내.pdf) → Agent 계획 → Tool 실행(B-2002) → 모델 호출 → 응답 반환" 순서로 보입니다.

### 4.1. 모든 요청을 한 번에 재구성하기

Trace ID로 묶으면 요청마다 "누가, 어떤 문서를 근거로, 어떤 Tool을 불렀는지"를 한 줄로 요약할 수 있습니다. 사고 조사 화면이 보여 주는 것이 이런 요약입니다.

In [9]:
import re
from collections import defaultdict

by_trace = defaultdict(list)
for e in logs():
    by_trace[e["trace_id"]].append(e)
rows = []
for tid, es in by_trace.items():
    text = " ".join(e["msg"] for e in es)
    rows.append((tid, re.search(r"user=(\S+)", text).group(1), re.search(r"결과=(\[.*?\])", text).group(1),
                 re.search(r"arg=(\S+)", text).group(1), " → ".join(e["step"] for e in es)))
table(["Trace ID", "사용자", "근거 문서", "Tool 인자", "단계 순서"], rows)

| Trace ID | 사용자 | 근거 문서 | Tool 인자 | 단계 순서 |
|---|---|---|---|---|
| 256e3593ddf0 | park | ['교환정책.pdf#p2'] | C-3003 | gateway → rag → agent → tool → model → gateway |
| a756caa67ea0 | lee | ['배송안내.pdf#p1'] | B-2002 | gateway → rag → agent → tool → model → gateway |
| c46f87b1bb1b | kim | ['환불규정.pdf#p3'] | A-1001 | gateway → rag → agent → tool → model → gateway |

## 5. 로그에 남기기 전에 민감정보 가리기

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 설치 | ✅ 완료 |
| 1 | 서버 코드 살펴보고 실행하기 | ✅ 완료 |
| 2 | 여러 요청을 동시에 보내기 | ✅ 완료 |
| 3 | 로그 보기: Trace ID가 없다면? | ✅ 완료 |
| 4 | Trace ID로 요청 하나 재구성하기 | ✅ 완료 |
| 5 | 로그에 남기기 전에 민감정보 가리기 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 6 | 서비스 사이로 Trace ID 이어 주기 | 예정 |
| 7 | 체크포인트와 과제 | 예정 |
| 8 | 정리 | 예정 |

### 5.1. 마스킹이 켜진 상태

사용자는 질문에 연락처나 메일 주소를 그대로 적습니다. 이런 질문을 보내 봅니다.

In [10]:
PII_Q = "주문자 연락처 010-1234-5678, 메일 choi.yr@hanbit.example 인데 환불 되나요?"
t_on = ask("choi", PII_Q, "D-4004")
masked_on = [e for e in logs() if e["trace_id"] == t_on]
table(["단계", "메시지 (마스킹 켬)"], [(e["step"], e["msg"]) for e in masked_on])

| 단계 | 메시지 (마스킹 켬) |
|---|---|
| gateway | 요청 수신 user=choi question='주문자 연락처 010-****-5678, 메일 c***@hanbit.example 인데 환불 되나요?' |
| rag | 검색어='주문자 연락처 010-****-5678, 메일 c***@hanbit.example 인데 환불 되나요?' 결과=['환불규정.pdf#p3'] |
| agent | 계획: 문서 검색 → 주문 조회 Tool → 답변 생성 |
| tool | Tool 실행 name=get_order_status arg=D-4004 |
| model | 모델 호출 prompt_chars=113 tokens=152 |
| gateway | 응답 반환 |

### 5.2. 마스킹을 끄면

서버를 마스킹을 끈 채(`LOG_MASK=off`) 다시 띄우고 같은 질문을 보냅니다.

In [11]:
start_server(mask=False)
t_off = ask("choi", PII_Q, "D-4004")
masked_off = [e for e in logs() if e["trace_id"] == t_off]
SECRETS = ["010-1234-5678", "choi.yr@hanbit.example"]
table(["단계", "마스킹 켬: 원문 노출", "마스킹 끔: 원문 노출", "마스킹 끔 메시지"],
      [(a["step"], "있음" if any(s in a["msg"] for s in SECRETS) else "-",
        "**있음**" if any(s in b["msg"] for s in SECRETS) else "-", b["msg"][:70])
       for a, b in zip(masked_on, masked_off)])
start_server(mask=True)

서버 실행 중: PID 117489, 마스킹 끔


| 단계 | 마스킹 켬: 원문 노출 | 마스킹 끔: 원문 노출 | 마스킹 끔 메시지 |
|---|---|---|---|
| gateway | - | **있음** | 요청 수신 user=choi question='주문자 연락처 010-1234-5678, 메일 choi.yr@hanbit.exa |
| rag | - | **있음** | 검색어='주문자 연락처 010-1234-5678, 메일 choi.yr@hanbit.example 인데 환불 되나요?' 결과=[ |
| agent | - | - | 계획: 문서 검색 → 주문 조회 Tool → 답변 생성 |
| tool | - | - | Tool 실행 name=get_order_status arg=D-4004 |
| model | - | - | 모델 호출 prompt_chars=113 tokens=152 |
| gateway | - | - | 응답 반환 |

서버 실행 중: PID 117646, 마스킹 켬


두 가지를 확인합니다.

- **원문은 게이트웨이 로그에만 있지 않습니다.** 질문이 RAG의 검색어로도 쓰이므로 `rag` 로그에도 연락처가 그대로 남습니다. 로그를 남기는 곳마다 마스킹을 따로 넣으면 이런 곳을 빠뜨리기 쉽습니다. 이 서버는 모든 로그가 지나가는 한 곳(`JsonFormatter`)에서 가리므로, 새 로그를 추가해도 빠지지 않습니다. 실무에서는 이 역할을 AI Gateway나 로그 수집기가 맡습니다(9-2).
- **로그만 가리고 처리는 원문으로 합니다.** 모델과 Tool에는 원문이 그대로 전달되고, 가린 값은 로그 파일에만 남습니다. 주문번호(`D-4004`)나 금액처럼 사고 조사에 필요한 값은 규칙에 걸리지 않아 그대로 남습니다. 무엇을 가리고 무엇을 남길지는 조직의 개인정보 처리 기준에 맞춰 정합니다.

> 📌 (참고) `model` 로그에는 처음부터 프롬프트 원문 대신 글자 수와 토큰 수만 남깁니다. 마스킹 이전에 **원문을 아예 남기지 않는 것**이 가장 확실한 방법입니다.

## 6. 서비스 사이로 Trace ID 이어 주기

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 설치 | ✅ 완료 |
| 1 | 서버 코드 살펴보고 실행하기 | ✅ 완료 |
| 2 | 여러 요청을 동시에 보내기 | ✅ 완료 |
| 3 | 로그 보기: Trace ID가 없다면? | ✅ 완료 |
| 4 | Trace ID로 요청 하나 재구성하기 | ✅ 완료 |
| 5 | 로그에 남기기 전에 민감정보 가리기 | ✅ 완료 |
| 6 | 서비스 사이로 Trace ID 이어 주기 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 7 | 체크포인트와 과제 | 예정 |
| 8 | 정리 | 예정 |

실제 환경에서는 요청이 여러 서비스를 지나갑니다. 앞 서비스가 받은 Trace ID를 다음 서비스에 헤더로 넘겨야 끝까지 이어집니다. 이 서버는 `X-Trace-Id` 헤더를 받으면 그대로 이어 씁니다.

In [12]:
tid = ask("kim", "환불 되나요?", trace="gw-20261023-0001")   # 앞단(예: AI Gateway)이 만든 Trace ID
print("응답 헤더의 Trace ID:", tid)
print(sh(f"grep gw-20261023-0001 {LOG}"))

응답 헤더의 Trace ID: gw-20261023-0001
{"time": "05:28:01", "trace_id": "gw-20261023-0001", "step": "gateway", "msg": "요청 수신 user=kim question='환불 되나요?'"}
{"time": "05:28:01", "trace_id": "gw-20261023-0001", "step": "rag", "msg": "검색어='환불 되나요?' 결과=['환불규정.pdf#p3']"}
{"time": "05:28:02", "trace_id": "gw-20261023-0001", "step": "agent", "msg": "계획: 문서 검색 → 주문 조회 Tool → 답변 생성"}
{"time": "05:28:02", "trace_id": "gw-20261023-0001", "step": "tool", "msg": "Tool 실행 name=get_order_status arg=A-1001"}
{"time": "05:28:02", "trace_id": "gw-20261023-0001", "step": "model", "msg": "모델 호출 prompt_chars=61 tokens=152"}
{"time": "05:28:02", "trace_id": "gw-20261023-0001", "step": "gateway", "msg": "응답 반환"}


7장 실습 2의 접근 게이트웨이(PEP)가 만든 `trace` 값을 이 헤더로 넘기면, Gateway 로그와 서비스 로그가 하나의 ID로 연결됩니다. 실무에서는 W3C 표준 헤더(`traceparent`)와 OpenTelemetry를 써서 이 과정을 자동화합니다.

> 🙋 (생각해 보기)
> 이 서버는 받은 `X-Trace-Id`를 **검사 없이** 그대로 씁니다. 외부 사용자가 아주 긴 문자열이나 다른 사람의 Trace ID를 보내면 어떻게 될까요? 앞단 Gateway만 이 헤더를 넣을 수 있게 하고, 형식(길이, 허용 문자)을 검사해야 합니다.

## 7. 체크포인트와 과제

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 설치 | ✅ 완료 |
| 1 | 서버 코드 살펴보고 실행하기 | ✅ 완료 |
| 2 | 여러 요청을 동시에 보내기 | ✅ 완료 |
| 3 | 로그 보기: Trace ID가 없다면? | ✅ 완료 |
| 4 | Trace ID로 요청 하나 재구성하기 | ✅ 완료 |
| 5 | 로그에 남기기 전에 민감정보 가리기 | ✅ 완료 |
| 6 | 서비스 사이로 Trace ID 이어 주기 | ✅ 완료 |
| 7 | 체크포인트와 과제 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 8 | 정리 | 예정 |

### 7.1. 체크포인트

- [ ] Trace ID 없이 동시 요청의 로그를 구분하려면 무엇에 의존해야 했나요? (시간, 사용자 이름…) 그것만으로 충분했나요?
- [ ] Trace ID 하나로 요청의 전체 흐름(검색 문서, Tool, 모델 호출)을 재구성했나요?
- [ ] 마스킹을 껐을 때 연락처가 남은 로그는 어디였나요? 마스킹을 로그마다 따로 넣지 않고 한 곳에서 해야 하는 이유는 무엇인가요?
- [ ] 서비스 사이에 Trace ID를 넘기지 않으면 어디서 추적이 끊길까요?

### 7.2. 과제 1: 계좌번호 가리기 규칙 더하기

지금의 마스킹 규칙은 주민등록번호, 카드번호, 휴대전화 번호, 이메일만 가립니다. 6장 실습 2에서 본 것처럼 고객은 상담 중에 계좌번호도 말합니다. `trace_app.py`의 규칙을 그대로 읽어 와, `EXTRA`에 계좌번호 규칙을 더하고 시험합니다.

- 계좌번호(`국민은행 123456-01-789012`)는 가려야 합니다.
- 주문번호(`A-1001`), 날짜(`2026-10-23`), 금액(`35,000원`)은 사고 조사에 필요하므로 **남아야** 합니다.

In [13]:
# trace_app.py의 PATTERNS 정의를 그대로 읽어 옵니다 (서버 코드와 같은 규칙으로 시험하기 위해)
block = src[src.index("PATTERNS = ["):src.index("def mask")]
ns = {"re": re}
exec(block, ns)
PATTERNS = ns["PATTERNS"]

EXTRA = [
    # TODO: (re.compile(r"..."), r"...") 형식으로 계좌번호 규칙을 더합니다.
]


def mask2(text):
    for pattern, repl in PATTERNS + EXTRA:
        text = pattern.sub(repl, text)
    return text


TESTS = [
    ("환불 계좌는 국민은행 123456-01-789012입니다", "123456-01-789012", False),
    ("신한은행 110-123-456789로 보내 주세요", "110-123-456789", False),
    ("주문번호 A-1001, 2026-10-23 결제분 35,000원 환불 요청", "2026-10-23", True),
    ("연락처 010-1234-5678", "010-1234-5678", False),
]
table(["입력", "가린 결과", "기대", "판정"],
      [(t, mask2(t), "남아야 함" if keep else "가려야 함", "✅" if (s in mask2(t)) == keep else "❌") for t, s, keep in TESTS])

| 입력 | 가린 결과 | 기대 | 판정 |
|---|---|---|---|
| 환불 계좌는 국민은행 123456-01-789012입니다 | 환불 계좌는 국민은행 123456-01-789012입니다 | 가려야 함 | ❌ |
| 신한은행 110-123-456789로 보내 주세요 | 신한은행 110-123-456789로 보내 주세요 | 가려야 함 | ❌ |
| 주문번호 A-1001, 2026-10-23 결제분 35,000원 환불 요청 | 주문번호 A-1001, 2026-10-23 결제분 35,000원 환불 요청 | 남아야 함 | ✅ |
| 연락처 010-1234-5678 | 연락처 010-****-5678 | 가려야 함 | ✅ |

<details>
<summary>💡 힌트 보기</summary>

숫자 묶음 모양(`\d+-\d+-\d+`)만으로 찾으면 날짜 `2026-10-23`까지 가려집니다. 계좌번호 앞에 은행 이름이 오는 경우가 많다는 **맥락**을 함께 씁니다.

```python
EXTRA = [
    (re.compile(r"(은행\s*)(\d{2,6})-(\d{2,6})-(\d{2,8})"), r"\1\2-***-****"),
]
```

은행 이름 없이 숫자만 말한 계좌번호는 여전히 놓칩니다. 그래서 6장 실습 2와 같은 결론에 닿습니다. 규칙은 계속 늘려야 하는 목록이고, 로그에는 **원문을 처음부터 덜 남기는 것**이 먼저입니다. 규칙을 서버에 반영하려면 `trace_app.py`의 `PATTERNS`에 같은 줄을 더하고 서버를 다시 시작합니다.
</details>

### 7.3. 과제 2: 로그 보관과 열람 기준 (서술)

마스킹을 해도 로그에는 사용자 이름, 질문 내용, 주문번호가 남습니다. 아래 질문에 답해 보세요.

1. 이 로그는 누가 볼 수 있어야 하고, 얼마 동안 보관해야 할까요? (사고 조사에 필요한 기간과 개인정보 최소 보관의 균형)
2. 사고 조사를 위해 원문이 꼭 필요한 경우가 있다면, 원문은 어디에 어떤 통제로 남겨야 할까요?

✍️ **답 적기** (이 셀을 더블클릭해 고칩니다)

1.
2.

## 8. 정리

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 설치 | ✅ 완료 |
| 1 | 서버 코드 살펴보고 실행하기 | ✅ 완료 |
| 2 | 여러 요청을 동시에 보내기 | ✅ 완료 |
| 3 | 로그 보기: Trace ID가 없다면? | ✅ 완료 |
| 4 | Trace ID로 요청 하나 재구성하기 | ✅ 완료 |
| 5 | 로그에 남기기 전에 민감정보 가리기 | ✅ 완료 |
| 6 | 서비스 사이로 Trace ID 이어 주기 | ✅ 완료 |
| 7 | 체크포인트와 과제 | ✅ 완료 |
| 8 | 정리 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |

In [14]:
server.terminate()
server.wait(timeout=5)
print(sh("rm -rf ~/trace-lab ~/trace-venv && echo '서버 종료, 실습 폴더와 가상환경 삭제됨'"))

서버 종료, 실습 폴더와 가상환경 삭제됨


---
> 다음 실습 3에서는 서버 안이 아니라 클라우드 관리형 서비스의 버전을 지원 종료 일정과 자동으로 대조합니다.